<a href="https://colab.research.google.com/github/SoumayaMAMI/Fine-tuning-llama-3-LoRa-/blob/main/Data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Dataset inspection**


### **Loading the RecipeNLG Dataset**

In this step, we load the RecipeNLG dataset directly from its online source using the Hugging Face Datasets library.

We use streaming mode to retrieve only the first 1,000 recipes without downloading the entire dataset.


In [ ]:
!pip install -q datasets

from datasets import load_dataset

# Direct URL of the RecipeNLG dataset
url = "https://huggingface.co/datasets/Mahimas/recipenlg/resolve/main/RecipeNLG.csv"

# Load the dataset in streaming mode
dataset = load_dataset(
    "csv",
    data_files=url,
    split="train",
    streaming=True
)

# Select 1000 recipes
dataset_1000 = list(dataset.take(1000))

print("Number of recipes:", len(dataset_1000))
print("\nFirst recipe:")
print(dataset_1000[0])

Number of recipes: 1000

First recipe:
{'Unnamed: 0': 0, 'title': 'No-Bake Nut Cookies', 'ingredients': '["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]', 'directions': '["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]', 'link': 'www.cookbooks.com/Recipe-Details.aspx?id=44874', 'source': 'Gathered', 'NER': '["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]'}


### **Dataset Dimensions**
We examine the dimensions of the dataset to determine the number of recipes and the number of available features.

In [ ]:
#Import pandas for dataset inspection
import pandas as pd

#Convert the list of recipes into a Dataframe
df = pd.DataFrame(dataset_1000)

#Display the dataset dimensions
print("Number of rows:", df.shape[0]) # number of recipes
print("Number of columns:", df.shape[1]) # number of features

Number of rows: 1000
Number of columns: 7


**Observation:** The dataset contains 1000 recipes and 7 columns.


### **Dataset Structure**

We inspect the dataset structure to identify the available columns, their data types, and the number of non-null values in each column.

In [ ]:
#Display all column names
print("Dataset columns:", df.columns.tolist())

Dataset columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']


In [ ]:
#Display the data types
print("\nDataset information:")
df.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   title        1000 non-null   object
 2   ingredients  1000 non-null   object
 3   directions   1000 non-null   object
 4   link         1000 non-null   object
 5   source       1000 non-null   object
 6   NER          1000 non-null   object
dtypes: int64(1), object(6)
memory usage: 54.8+ KB


**Observations:**

- The dataset contains 7 columns and 1,000 entries.
- All columns contain 1,000 non-null values which means no missing values were detected in the dataset at this stage.
- One column (`Unnamed: 0`) has an integer data type.
- The remaining six columns have an object data type.
- No missing values were detected in the dataset at this stage.

In [ ]:
# Inspect the Unnamed: 0 column
print(df["Unnamed: 0"].head())
print("Number of unique values:", df["Unnamed: 0"].nunique())

0    0
1    1
2    2
3    3
4    4
Name: Unnamed: 0, dtype: int64
Number of unique values: 1000


**Observation:** The Unnamed: 0 column appears to contain an index or identifier associated with each recipe.

In [ ]:
#Check for completely duplicated recipes
duplicate_count = df.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)

Number of duplicated rows: 0


**Observation:** No completely duplicated rows were detected in the selected 1,000 recipes.


### **Recipe Content Inspection**
We inspect the content of the main recipe columns to understand how recipe information is represented in the dataset.

In [ ]:
print("Title:", df["title"].iloc[0])
print("Ingredients:", df["ingredients"].iloc[0])
print("Directions:", df["directions"].iloc[0])
print("NER:", df["NER"].iloc[0])

Title: No-Bake Nut Cookies
Ingredients: ["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]
Directions: ["In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine.", "Stir over medium heat until mixture bubbles all over top.", "Boil and stir 5 minutes more. Take off heat.", "Stir in vanilla and cereal; mix well.", "Using 2 teaspoons, drop and shape into 30 clusters on wax paper.", "Let stand until firm, about 30 minutes."]
NER: ["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]


**Observations:**
- The `title` column contains the recipe names.
- The `ingredients` column contains the ingredients required for each recipe.
- The `directions` column contains the preparation instructions.
- The `NER` column contains named entities extracted from the recipe content.

**Conclusion:**
The inspection provides an initial understanding of the recipe content and these columns contain the main information needed to represent recipes as instruction-response pairs in the following stages.

## DATASET CLEANING 

Reusable functions are used in this section so that the cleaning pipeline can be applied to the same 1,000 examples or to additional samples in the future.

### Cleaning and Preparation Tasks

- Cleaning whitespace and text formatting
- Parsing ingredients and preparation directions
- Formatting the recipe content consistently
- Creating instruction-response pairs for fine-tuning
- Validating the number of prepared examples
- Exporting the final cleaned dataset


In [20]:
import ast
import re
import pandas as pd

In [21]:
def clean_text(value):
    """Remove extra whitespace and convert missing values to empty text."""
    if pd.isna(value):
        return ""

    value = str(value).strip()
    return re.sub(r"\s+", " ", value)

def parse_recipe_list(value):
    """Parse RecipeNLG list-like strings and return clean list items."""
    value = clean_text(value)

    if not value:
        return []

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return [
                clean_text(item)
                for item in parsed
                if clean_text(item)
            ]
    except (ValueError, SyntaxError):
        pass

    return [value]


def format_ingredients(value):
    ingredients = parse_recipe_list(value)

    if ingredients:
        return "\n".join(f"- {item}" for item in ingredients)

    return "No ingredients provided."


def format_directions(value):
    directions = parse_recipe_list(value)

    if directions:
        return "\n".join(
            f"{number}. {step}"
            for number, step in enumerate(directions, start=1)
        )

    return "No preparation directions provided."


def prepare_recipe_samples(
    data,
    expected_count=1000,
    save_path="cleaned_recipes.csv"
):
    """
    Prepare RecipeNLG recipes for instruction-fine-tuning.

    Parameters
    ----------
    data : pandas.DataFrame or list of dictionaries
        The recipes loaded by Hugging Face.
    expected_count : int or None
        Required number of samples. Use None to disable the count check.
    save_path : str or None
        Save the result to CSV. Use None if saving is not required.

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing exactly two columns: instruction and response.
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        df = pd.DataFrame(data)

    if expected_count is not None and len(df) != expected_count:
        raise ValueError(
            f"Expected exactly {expected_count} examples, "
            f"but received {len(df)}."
        )

    for column in [
        "title",
        "ingredients",
        "directions",
        "link",
        "source",
        "NER"
    ]:
        if column not in df.columns:
            df[column] = ""
        else:
            df[column] = df[column].apply(clean_text)

    df["ingredients_clean"] = df["ingredients"].apply(format_ingredients)
    df["directions_clean"] = df["directions"].apply(format_directions)
    df["NER_clean"] = df["NER"].apply(
        lambda value: ", ".join(parse_recipe_list(value))
    )

    df["instruction"] = df.apply(
        lambda row:
            f"Create the recipe '{row['title']}'. "
            f"Use these ingredients:\n{row['ingredients_clean']}",
        axis=1
    )

    df["response"] = df.apply(
        lambda row:
            f"{row['title']}\n\n"
            "Ingredients:\n"
            f"{row['ingredients_clean']}\n\n"
            "Directions:\n"
            f"{row['directions_clean']}",
        axis=1
    )

    df_final = df[["instruction", "response"]].copy()

    if expected_count is not None and len(df_final) != expected_count:
        raise RuntimeError(
            f"Prepared dataset size changed: expected {expected_count} examples, "
            f"but received {len(df_final)}."
        )

    if save_path is not None:
        df_final.to_csv(save_path, index=False)
        print(f"Saved {len(df_final)} examples to {save_path}")

    print(f"Prepared {len(df_final)} examples.")
    return df_final

To prepare the current 1,000 examples:

### Preparing the Training Examples

The reusable cleaning pipeline is applied to the 1,000 original RecipeNLG samples. It converts the recipe fields into formatted instruction-response pairs and saves the final dataset as a CSV file.

In [22]:
df = pd.DataFrame(dataset_1000)

df_final = prepare_recipe_samples(
    df,
    expected_count=1000,
    save_path="cleaned_recipes.csv"
)

Saved 1000 examples to cleaned_recipes.csv
Prepared 1000 examples.


### Final Dataset Validation

The following checks verify that the prepared dataset contains exactly 1,000 examples, has the required columns, and contains no missing or empty text values.

The cleaning pipeline also checks for duplicate instruction-response pairs and reloads the saved CSV file to confirm that the export is readable and complete.

In [ ]:

if "df_final" not in globals():
    raise NameError(
        "df_final is not defined. Call prepare_recipe_samples() before running this code."
    )

expected_count = 1000

print("=== Dataset size verification ===")
print(f"Number of examples: {len(df_final)}")

if len(df_final) != expected_count:
    raise RuntimeError(
        f"Expected {expected_count} examples, but found {len(df_final)}."
    )

print("Dataset size check passed.")


required_columns = ["instruction", "response"]

missing_columns = [
    column for column in required_columns
    if column not in df_final.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing required columns: {missing_columns}"
    )

print("\n=== Required column verification ===")
print("Required columns are present.")


missing_values = df_final[required_columns].isna().sum()

print("\n=== Missing values ===")
print(missing_values)

if missing_values.sum() != 0:
    raise RuntimeError(
        "One or more instruction-response values are missing."
    )

print("Missing-value check passed.")

empty_instructions = (
    df_final["instruction"].astype(str).str.strip() == ""
).sum()

empty_responses = (
    df_final["response"].astype(str).str.strip() == ""
).sum()

print("\n=== Empty text verification ===")
print(f"Empty instructions: {empty_instructions}")
print(f"Empty responses: {empty_responses}")

if empty_instructions != 0 or empty_responses != 0:
    raise RuntimeError("The dataset contains empty instructions or responses.")

print("Empty-text check passed.")


duplicate_count = df_final.duplicated(
    subset=["instruction", "response"]
).sum()

print("\n=== Duplicate verification ===")
print(f"Duplicate instruction-response pairs: {duplicate_count}")


csv_path = "cleaned_recipes.csv"


df_saved = pd.read_csv(csv_path)

print("\n=== Saved CSV verification ===")
print(f"Saved file shape: {df_saved.shape}")

if len(df_saved) != expected_count:
    raise RuntimeError(
        f"The saved CSV does not contain {expected_count} examples."
    )

print("Saved CSV size check passed.")

=== Dataset size verification ===
Number of examples: 1000
Dataset size check passed.

=== Required column verification ===
Required columns are present.

=== Missing values ===
instruction    0
response       0
dtype: int64
Missing-value check passed.

=== Empty text verification ===
Empty instructions: 0
Empty responses: 0
Empty-text check passed.

=== Duplicate verification ===
Duplicate instruction-response pairs: 0

=== Saved CSV verification ===
Saved file shape: (1000, 2)
Saved CSV size check passed.


### Preview of the Cleaned CSV

The first five rows show the final instruction-response format and confirm that the cleaned CSV contains the expected columns.

In [24]:
print("Displayed CSV shape:")
print(df_saved.shape)

df_saved.head()

Displayed CSV shape:
(1000, 2)


,instruction,response
0,Create the recipe 'No-Bake Nut Cookies'. Use t...,No-Bake Nut Cookies\n\nIngredients:\n- 1 c. fi...
1,Create the recipe 'Jewell Ball'S Chicken'. Use...,Jewell Ball'S Chicken\n\nIngredients:\n- 1 sma...
2,Create the recipe 'Creamy Corn'. Use these ing...,Creamy Corn\n\nIngredients:\n- 2 (16 oz.) pkg....
3,Create the recipe 'Chicken Funny'. Use these i...,Chicken Funny\n\nIngredients:\n- 1 large whole...
4,Create the recipe 'Reeses Cups(Candy)'. Use th...,Reeses Cups(Candy)\n\nIngredients:\n- 1 c. pea...


### Inspection of a Recipe Example

This example shows the complete instruction and response for one recipe. It verifies that the ingredients and numbered preparation directions are correctly formatted.

In [27]:
example = df_saved.iloc[1]

print("=== Instruction ===")
print(example["instruction"])

print("\n=== Response ===")
print(example["response"])

=== Instruction ===
Create the recipe 'Jewell Ball'S Chicken'. Use these ingredients:
- 1 small jar chipped beef, cut up
- 4 boned chicken breasts
- 1 can cream of mushroom soup
- 1 carton sour cream

=== Response ===
Jewell Ball'S Chicken

Ingredients:
- 1 small jar chipped beef, cut up
- 4 boned chicken breasts
- 1 can cream of mushroom soup
- 1 carton sour cream

Directions:
1. Place chipped beef on bottom of baking dish.
2. Place chicken on top of beef.
3. Mix soup and cream together; pour over chicken. Bake, uncovered, at 275° for 3 hours.


### Random Sample Inspection

These randomly selected records provide a representative preview of the cleaned training data.

In [26]:
df_saved.sample(
    n=5,
    random_state=42
)

,instruction,response
521,Create the recipe 'Pinto Beans'. Use these ing...,Pinto Beans\n\nIngredients:\n- 1 1/2 c. dry be...
737,Create the recipe 'Corn Light Bread'. Use thes...,Corn Light Bread\n\nIngredients:\n- 2 c. self-...
740,Create the recipe 'D'S Danish'. Use these ingr...,D'S Danish\n\nIngredients:\n- 4 c. flour\n- 3 ...
660,Create the recipe 'Carrot Cake'. Use these ing...,Carrot Cake\n\nIngredients:\n- 3 c. flour\n- 2...
411,"Create the recipe 'Low-Fat ""French Fries""'. Us...","Low-Fat ""French Fries""\n\nIngredients:\n- 1 1/..."


### Conclusion

The first 1,000 RecipeNLG examples were cleaned and formatted as instruction-response pairs. Validation confirmed that the final dataset contains exactly 1,000 examples, includes the required columns, and has no missing or empty instruction-response values. The cleaned data was saved as a CSV and is ready for the next stage: the train/test split and Llama 3 8B LoRA fine-tuning.